In [ ]:
# The simplest way to use structured outputs is with client.responses.parse() and a Pydantic model passed via text_format. 
# Pydantic handles the conversion from your Python class to a JSON schema automatically.

# 1. Named Entity Extraction Program 
"""
In this example we define a Person model and a PeopleList wrapper, 
then pass a block of unstructured text to the model. The model extracts every person it finds and returns them as typed Pydantic objects.
"""
from pydantic import BaseModel
from typing import Optional
from llm_config import ollama ,MODEL_OLLAMA
text = """
Marie Curie, born on November 7, 1867 in Warsaw, Poland, was a pioneering physicist 
and chemist who conducted groundbreaking research on radioactivity. She was the first 
woman to win a Nobel Prize. Her colleague Albert Einstein, a German-born theoretical 
physicist born on March 14, 1879, developed the theory of relativity. Meanwhile, 
Ada Lovelace, an English mathematician born on December 10, 1815, is often regarded 
as the first computer programmer for her work on Charles Babbage's Analytical Engine.
"""

class Person(BaseModel):
    name : str
    date_of_birth : str
    occupation : str
    nationality : str 

class peoplelist(BaseModel):
    people : list[Person]

response = ollama.responses.parse( 
    model = MODEL_OLLAMA,
    input = [{"role" : "developer", "content": "extract all people information mentioned in the para"},
             {"role" : "user", "content" : text}],
    text_format  = peoplelist
)

"""
response                                              <------------Object-------------
  └── output_parsed                             
       └── people                                     <------------list
            └── [0]                                   ← first Person
                 ├── name
                 ├── date_of_birth
                 ├── occupation
                 └── nationality

response → object → use .
output_parsed → object → use .
people → list → use [0]
first Person → object → use .name
"""


In [ ]:
print(response.output_parsed.people)
print(response.output_parsed.people[0])
print(response.output_parsed.people[0].occupation)    <---- . means "give me something inside this object. output_parsed and people are attributes of response.

[Person(name='Marie Curie', date_of_birth='November 7, 1867', occupation='physicist', nationality='Polish')]
name='Marie Curie' date_of_birth='November 7, 1867' occupation='physicist' nationality='Polish'
physicist


In [28]:
if response.output_parsed:
    people = response.output_parsed.people
else:
    people = []
for ppl in people:
    print(f"{ppl.name} | {ppl.date_of_birth} | {ppl.occupation} | {ppl.nationality}")

Marie Curie | November 7, 1867 | physicist | Polish


In [33]:
#-------------Web Search + Structured Outputs
# combine the web_search tool with structured outputs to search the web for live information and extract it into a clean schema.
# When using tools alongside structured outputs, we use client.responses.create() with the text format parameter instead of responses.parse(). 
# This requires us to build the JSON schema manually from the Pydantic model and set 
#   strict: True and 
#   additionalProperties: False 
# on all objects in the schema.

class NewsArticle(BaseModel):
    title : str
    date : str
    summary : str
    source : str

class site(BaseModel):
    articles : list[NewsArticle]

In [ ]:
response = ollama.responses.parse(
    model = MODEL_OLLAMA, 
    tools = [{"type": "web_search"}],                ## This is standard and ollama maynot support it on local setup.
    input = "Find 3 recent positive news stories from this week about technology or science",
    text_format = site
)

print(response.output_parsed.articles[0])